# K2 · Optimized candidate recipe + post-training evaluation

**Recipe mới để thử, chưa chứng minh tăng điểm và không hứa vượt80%.** Giữ nguyên B0; tăng canvas640×1280, warmup500, crop ưu tiên lớp ít xuất hiện, 40epoch,4workers; K1 dùng50% ảnh sạch và severity1–3. Mặc định chạy baseline mới trước.

Gắn **Cityscapes + ACDC**, bật **GPU + Internet**. Train chỉ Cityscapes train2975, chọn best trên Cityscapes val500. Sau khi đủ40epoch và khóa checkpoint, tự đánh giá Cityscapes val500 + ACDC val406, xuất PNG/SVG và ZIP. ACDC test chính thức giấu nhãn, nên đây là đánh giá validation có nhãn, không phải điểm hidden test.

Chạy một ARM mỗi phiên. Baseline/K2 phải cùng recipe mới; không quy chênh lệch so với run20epoch cũ cho FiLM. Thời gian/VRAM cần đo thực tế, có thể phải resume nhiều phiên. Checkpoint40epoch không được tạo bằng cách sửa horizon của ZIP20epoch. K3/K4 chưa bật.

In [ ]:
"""Standard-library source bootstrap shared by the two DG model notebooks."""
from pathlib import Path, PurePosixPath
import shutil
import stat
import zipfile


MARKERS=('scripts/run_dg_suite.py','src/training/checkpoints.py','src/models/registry.py')


def valid_source(path):
    path=Path(path)
    if path.is_dir():return all((path/n).is_file() for n in MARKERS)
    if path.is_file() and zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:return set(MARKERS).issubset(z.namelist())
    return False


def resolve_source(hint,input_root):
    hint=Path(hint);input_root=Path(input_root)
    if valid_source(hint):return hint
    search_root=hint if hint.is_dir() and hint.resolve()!=input_root.resolve() else input_root
    anchor=PurePosixPath(MARKERS[0])
    folders=sorted({p.parents[len(anchor.parts)-1] for p in search_root.rglob(anchor.name) if valid_source(p.parents[len(anchor.parts)-1])})
    candidates=folders or [p for p in sorted(search_root.rglob('*.zip')) if valid_source(p)]
    if len(candidates)!=1:raise ValueError(f'Need exactly one current code source; found{len(candidates)}: {candidates}. Set CODE_INPUT explicitly.')
    return candidates[0]


def extract_safe(archive,destination):
    destination=Path(destination).resolve();destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        seen=set()
        for item in z.infolist():
            name=PurePosixPath(item.filename.replace('\\','/'))
            if name.is_absolute() or '..' in name.parts or ':' in str(name) or stat.S_IFMT(item.external_attr>>16)==stat.S_IFLNK:
                raise ValueError('Unsafe archive member')
            key=str(name).rstrip('/').casefold()
            if key in seen:raise ValueError('Duplicate archive member')
            seen.add(key)
            if not (destination/str(name)).resolve().is_relative_to(destination):raise ValueError('Unsafe archive destination')
        z.extractall(destination)


def copy_source(source,destination):
    source=Path(source);destination=Path(destination)
    if not valid_source(source):raise ValueError('Stale or missing suite code')
    if source.is_file():extract_safe(source,destination)
    else:
        destination.mkdir(parents=True,exist_ok=True)
        for name in ('src','scripts','configs','tests','third_party'):
            if (source/name).exists():shutil.copytree(source/name,destination/name,dirs_exist_ok=True)
        for name in ('requirements.txt','pyproject.toml','THIRD_PARTY_NOTICES.md'):
            if (source/name).is_file():shutil.copy2(source/name,destination/name)

import os, sys, json, hashlib, subprocess, time
STARTED = time.monotonic()
ARM = 'baseline'  # baseline, film_clean, aug, film, k2. One arm per fresh session.
SEED = 42
EPOCHS = 40
BATCH_SIZE = 2
WEATHER_MODE = 'no-fog'  # Match the completed four-condition pilot; full needs metric depth for every train image.
DEPTH_ROOT = None  # Optional root containing disparity/train and camera/train (may be CITYSCAPES_ROOT)
DISTANCE_ROOT = None  # Alternative folder of <original-image-stem>.npy camera-ray meters
INVALID_POLICY = 'far'  # Explicit approximation: missing stereo pixels -> 200m, or use 'error'
CITYSCAPES_ROOT = None  # Folder containing leftImg8bit/{train,val} and gtFine/{train,val}
CODE_INPUT = None  # None clones the branch; alternatively set to your uploaded project-code.zip/folder
RESUME_INPUT = None  # Previous k2_<arm>_seed42.zip or its Kaggle-expanded folder
GIT_REF = 'feat/k1-weather-augmentation'
RECIPE = 'balanced_v2'
EVAL_AFTER_TRAIN = True
ACDC_ROOT = None  # Auto-find rgb_anon/{fog,night,rain,snow}/val AFTER checkpoint freeze.
ACDC_MASK_ROOT = None  # Set separately only if gt/ is mounted in another dataset.
BASE = Path('/kaggle/working')
WORK = BASE / f'k2_balanced_v2_{ARM}_seed{SEED}'
REPO = BASE / f'k2_balanced_v2_code_{ARM}_seed{SEED}'
ARCHIVE = BASE / f'k2_balanced_v2_{ARM}_seed{SEED}.zip'
SHARE_ARCHIVE = BASE / f'k2_balanced_v2_{ARM}_seed{SEED}_share.zip'
assert not WORK.exists() and not REPO.exists(), 'Use a fresh session; preserve previous ZIP first'
if RESUME_INPUT is not None:
    previous = Path(RESUME_INPUT)
    if previous.is_file():
        extract_safe(previous, WORK)
    else:
        found = [previous] if (previous/'k2_plan.json').is_file() else [p.parent for p in previous.rglob('k2_plan.json')]
        assert len(found) == 1, f'Set RESUME_INPUT to one K2 result: {found}'
        shutil.copytree(found[0], WORK)
    plan = json.loads((WORK/'k2_plan.json').read_text())
    assert plan['protocol'] == 'weatherseg_k2_source_v1'
    assert plan.get('recipe','legacy_v1') == RECIPE, 'Use notebook04 to resume a legacy run'
    assert (plan['arm'],plan['seed'],plan['epochs'],plan['batch_size']) == (ARM,SEED,EPOCHS,BATCH_SIZE)
    original = plan['weather_inputs']
    WEATHER_MODE, DEPTH_ROOT, DISTANCE_ROOT, INVALID_POLICY = (
        original['mode'], original['depth_root'], original['distance_root'], original['invalid_policy'])
    copy_source(WORK/'source.zip' if (WORK/'source.zip').is_file() else WORK/'source', REPO)
    assert all(hashlib.sha256((REPO/n).read_bytes()).hexdigest() == h for n,h in plan['source_hashes'].items())
    CITYSCAPES_ROOT = Path(plan['city_root'])
elif CODE_INPUT is not None:
    copy_source(resolve_source(Path(CODE_INPUT), Path('/kaggle/input')), REPO)
else:
    subprocess.run(['git','clone','--depth','1','--single-branch','--branch',GIT_REF,
                    'https://github.com/MinhToan1001/adverse-weather-segmentation.git',str(REPO)],check=True)
assert (REPO/'scripts/run_k2.py').is_file(), 'Need the K2 branch/code bundle'
if CITYSCAPES_ROOT is None:
    found = sorted({p.parent for p in Path('/kaggle/input').rglob('leftImg8bit') if p.is_dir()
                    and all((p.parent/n).is_dir() for n in ['leftImg8bit/train','gtFine/train','leftImg8bit/val','gtFine/val'])})
    assert len(found) == 1, f'Set CITYSCAPES_ROOT explicitly: {found}'
    CITYSCAPES_ROOT = found[0]
os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ.update(USE_TF='0',USE_TORCH='1',OMP_NUM_THREADS='2',MKL_NUM_THREADS='2')
print('Selected arm:', ARM, '| seed:', SEED, '| epochs:', EPOCHS)
print('TRAIN: Cityscapes train2975. SELECT BEST: Cityscapes val500. ACDC: evaluation later, never training here.')
print('Cityscapes:', CITYSCAPES_ROOT, '| code:', REPO)
print('This notebook runs ONE arm. Other arms require a fresh session and independent ImageNet initialization.')
print('Source snapshot recipe is restored on resume; do not change ARM/SEED/EPOCHS/BATCH_SIZE.')

EVAL_ROOT = BASE / f'k2_balanced_v2_{ARM}_seed{SEED}_evaluation'
EVAL_ARCHIVE = BASE / f'k2_balanced_v2_{ARM}_seed{SEED}_evaluation.zip'
FIGURES = WORK / 'figures'
print('Candidate recipe:', RECIPE, '| 640x1280 | warmup500 | source class-aware crop | cleaner K1')
print('No accuracy gain is guaranteed. Compare all arms under this SAME recipe/horizon.')


In [ ]:
def run(*args):
    subprocess.run([sys.executable,'-u',*map(str,args)],check=True,cwd=REPO)
run('-m','pip','install','transformers==4.57.6','numpy>=1.26,<3','Pillow>=10,<12',
    'PyYAML>=6,<7','matplotlib>=3.8,<4','safetensors>=0.4')
import torch
import torchvision
assert torch.cuda.is_available(), 'Enable a GPU accelerator'
assert tuple(map(int,torch.__version__.split('+')[0].split('.')[:2])) >= (2,6), 'Need Torch>=2.6'
print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))


In [ ]:
# Tiny random-init CUDA/AMP check, not a quality benchmark or pretrained training.
run('-m','scripts.verify_k2','--device','cuda','--output',BASE/'k2_smoke.json')
print((BASE/'k2_smoke.json').read_text())


In [ ]:
from src.experiments.packaging import export_workspace
from IPython.display import FileLink, display
remaining = 9 - (time.monotonic() - STARTED)/3600
weather_args = ['--weather-mode',WEATHER_MODE,'--invalid-policy',INVALID_POLICY]
if DEPTH_ROOT is not None:
    weather_args += ['--depth-root',str(DEPTH_ROOT)]
if DISTANCE_ROOT is not None:
    weather_args += ['--distance-root',str(DISTANCE_ROOT)]
try:
    if remaining > 0:
        run('-m','scripts.run_k2','--city',CITYSCAPES_ROOT,'--root',WORK,'--archive',ARCHIVE,
            '--recipe',RECIPE,'--arm',ARM,'--seed',SEED,'--epochs',EPOCHS,'--batch-size',BATCH_SIZE,
            '--session-hours',remaining,*weather_args)
    else:
        print('Setup exhausted this session window; keep inputs and continue in a fresh session.')
finally:
    if WORK.exists() and not ARCHIVE.is_file():
        export_workspace(WORK,ARCHIVE)
    if ARCHIVE.is_file():
        print('FULL ZIP for resume:',ARCHIVE)
        display(FileLink(str(ARCHIVE)))
    # A legacy result restores its old runner; create a share ZIP below without changing that source.
    if SHARE_ARCHIVE.is_file():
        print('SMALL ZIP for review/demo (cannot resume):',SHARE_ARCHIVE)
        display(FileLink(str(SHARE_ARCHIVE)))


In [ ]:
# Source diagnostics first; target data is read only for a COMPLETE sealed candidate.
if (WORK/'run/history.jsonl').is_file():
    run('-m','scripts.plot_k2','--input',WORK,'--out',FIGURES)
state = json.loads((WORK/'status.json').read_text()) if (WORK/'status.json').is_file() else {}
try:
    if EVAL_AFTER_TRAIN and state.get('status') == 'training_complete':
        args = ['-m','scripts.evaluate_k2','--input',WORK,'--out',EVAL_ROOT,
                '--city',CITYSCAPES_ROOT,'--input-root','/kaggle/input','--device','cuda']
        if ACDC_ROOT is not None: args += ['--acdc',str(ACDC_ROOT)]
        if ACDC_MASK_ROOT is not None: args += ['--acdc-masks',str(ACDC_MASK_ROOT)]
        run(*args)
    else:
        print('Target evaluation not run:',state.get('status'),'| EVAL_AFTER_TRAIN:',EVAL_AFTER_TRAIN)
        print('Resume unfinished training with its full ZIP. Use notebook06 later for evaluation only.')
finally:
    # Keep trained checkpoints and diagnostics even if a target dataset is missing or evaluation fails.
    try:
        if (WORK/'run/history.jsonl').is_file():
            args = ['-m','scripts.plot_k2','--input',WORK,'--out',FIGURES]
            if EVAL_ROOT.exists(): args += ['--evaluation',str(EVAL_ROOT)]
            run(*args)
            run('-m','scripts.k2_results','share','--workspace',WORK,'--out',SHARE_ARCHIVE)
    finally:
        if WORK.exists(): export_workspace(WORK,ARCHIVE)
        if EVAL_ROOT.exists(): export_workspace(EVAL_ROOT,EVAL_ARCHIVE)
    for path in [ARCHIVE,SHARE_ARCHIVE,EVAL_ARCHIVE]:
        if path.is_file(): print('OUTPUT:',path); display(FileLink(str(path)))
from IPython.display import Image as NotebookImage, display, FileLink
for name in ['training_pipeline','k2_architecture','learning_curves','class_learning','best_class_iou','condition_learning','weather_miou']:
    path = FIGURES / (name+'.png')
    if path.is_file(): display(NotebookImage(filename=str(path)))
if EVAL_ROOT.exists():
    for path in sorted(EVAL_ROOT.glob('*/panels/*.png')):
        display(NotebookImage(filename=str(path)))


Nếu train chưa đủ40epoch, dùng ZIP đầy đủ để resume. Nếu train xong nhưng evaluation thiếu dữ liệu/hết phiên, dùng notebook06 với ZIP đã lưu để chỉ evaluate, không train lại. ZIP share chứa best model + log + biểu đồ; ZIP evaluation chứa metrics, confusion và ảnh dự đoán. Điểm ACDC được xem trong phát triển phải ghi là exploratory; không tune theo ACDC rồi gọi là test độc lập.